# 04 Data enrichment
Derive flow features, check single-feature leakage on TRAIN, and fit preprocessing on TRAIN only.

In [ ]:
import sys, json, ipaddress
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from src import config as cfg
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
import joblib

train = pd.read_csv(cfg.DATA_PROCESSED / "train.csv")
test = pd.read_csv(cfg.DATA_PROCESSED / "test.csv")
for d in (train, test): d[cfg.TIME] = pd.to_datetime(d[cfg.TIME])

def safe_div(num, den):
    num, den = pd.to_numeric(num, errors="coerce"), pd.to_numeric(den, errors="coerce")
    return (num / den).where(den > 0, 0).where(num.notna() & den.notna(), np.nan)

def port_range(values):
    values = pd.to_numeric(values, errors="coerce")
    return pd.Series(np.select([values.between(0, 1023), values.between(1024, 49151), values >= 49152], ["well_known", "registered", "dynamic"], default="missing"), index=values.index)

def add_features(d):
    d = d.copy(); total = d[cfg.BYTES_SENT] + d[cfg.BYTES_RECEIVED]
    d["total_bytes"] = total
    d["total_bytes_per_packet"] = safe_div(total, d[cfg.PACKETS])
    d["out_in_byte_ratio"] = safe_div(d[cfg.BYTES_SENT], d[cfg.BYTES_RECEIVED])
    d["packets_per_sec"] = safe_div(d[cfg.PACKETS], d[cfg.DURATION])
    d["bytes_sent_per_sec"] = safe_div(d[cfg.BYTES_SENT], d[cfg.DURATION])
    d["bytes_received_per_sec"] = safe_div(d[cfg.BYTES_RECEIVED], d[cfg.DURATION])
    d["total_bytes_per_sec"] = safe_div(total, d[cfg.DURATION])
    d["is_zero_duration"] = (d[cfg.DURATION] == 0).astype(float)
    d["src_port_range"], d["dst_port_range"] = port_range(d[cfg.SRC_PORT]), port_range(d[cfg.DST_PORT])
    d["hour"], d["day_of_week"] = d[cfg.TIME].dt.hour.astype(float), d[cfg.TIME].dt.dayofweek.astype(float)
    d["is_weekend"] = (d["day_of_week"] >= 5).astype(float)
    d["src_is_private"] = d[cfg.SRC_IP].map(lambda v: float(ipaddress.ip_address(v).is_private) if pd.notna(v) else np.nan)
    d["dst_is_private"] = d[cfg.DST_IP].map(lambda v: float(ipaddress.ip_address(v).is_private) if pd.notna(v) else np.nan)
    src_net = d[cfg.SRC_IP].map(lambda v: ipaddress.ip_network(f"{v}/24", strict=False) if pd.notna(v) else None)
    dst_net = d[cfg.DST_IP].map(lambda v: ipaddress.ip_network(f"{v}/24", strict=False) if pd.notna(v) else None)
    d["same_subnet_24"] = (src_net == dst_net).astype(float).where(src_net.notna() & dst_net.notna(), np.nan)
    return d

train_e, test_e = add_features(train), add_features(test)
derived = ["total_bytes", "total_bytes_per_packet", "out_in_byte_ratio", "packets_per_sec", "bytes_sent_per_sec", "bytes_received_per_sec", "total_bytes_per_sec", "is_zero_duration", "src_port_range", "dst_port_range", "hour", "day_of_week", "is_weekend", "src_is_private", "dst_is_private", "same_subnet_24"]
doc = {c: ("derived from one flow", "safe zero/missing handling") for c in derived}
pd.DataFrame([{"feature": c, "formula": doc[c][0], "missing/zero handling": doc[c][1]} for c in derived]).to_csv(cfg.REPORTS / "feature_dictionary.csv", index=False)

## Leakage check
Flag single-feature power above 0.95 using TRAIN only.

In [ ]:
num_features = list(cfg.NUMERIC_COLS) + [c for c in derived if c not in ["src_port_range", "dst_port_range"]]
cat_features = [cfg.PROTOCOL, "src_port_range", "dst_port_range"]
if cfg.INCLUDE_IDENTIFIERS: cat_features += [cfg.SRC_IP, cfg.DST_IP]
y_train = train_e[cfg.LABEL].astype(int)
checks = []
for c in num_features:
    s = pd.to_numeric(train_e[c], errors="coerce").fillna(pd.to_numeric(train_e[c], errors="coerce").median())
    if s.nunique() > 1: checks.append((c, "numeric", max(roc_auc_score(y_train, s), 1 - roc_auc_score(y_train, s))))
for c in cat_features:
    s = train_e[c].astype("object").fillna("missing")
    checks.append((c, "categorical", pd.crosstab(s, y_train).max(axis=1).sum() / len(s)))
leakage = pd.DataFrame(checks, columns=["feature", "kind", "single_feature_power"]).sort_values("single_feature_power", ascending=False)
display(leakage)
print("Above 0.95:", leakage.loc[leakage.single_feature_power > .95, "feature"].tolist())

## Train-only preprocessing
Numeric data uses median imputation, train-derived skew decisions, optional log1p, and scaling.

In [ ]:
def model_frame(d):
    out = pd.DataFrame(index=d.index)
    for c in num_features: out[c] = pd.to_numeric(d[c], errors="coerce")
    for c in cat_features: out[c] = d[c].astype("object")
    return out

Xtr, Xte = model_frame(train_e), model_frame(test_e)
skewed = [c for c in num_features if Xtr[c].min() >= 0 and Xtr[c].nunique() > 2 and Xtr[c].skew() > 2]
plain = [c for c in num_features if c not in skewed]
log_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")), ("scale", StandardScaler())])
num_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
cat_pipe = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
blocks = [("num_log", log_pipe, skewed), ("num", num_pipe, plain), ("cat", cat_pipe, cat_features)]
pre = ColumnTransformer([(n, p, c) for n, p, c in blocks if c], remainder="drop", verbose_feature_names_out=False)
pre.fit(Xtr)
A_tr, A_te = pre.transform(Xtr), pre.transform(Xte)
names = list(pre.get_feature_names_out())
assert not np.isnan(A_tr).any() and not np.isnan(A_te).any()
pd.DataFrame(A_tr, columns=names, index=train_e["row_id"]).to_csv(cfg.DATA_PROCESSED / "X_train.csv")
pd.DataFrame(A_te, columns=names, index=test_e["row_id"]).to_csv(cfg.DATA_PROCESSED / "X_test.csv")
train_e[cfg.LABEL].set_axis(train_e["row_id"]).to_csv(cfg.DATA_PROCESSED / "y_train.csv", header=[cfg.LABEL])
test_e[cfg.LABEL].set_axis(test_e["row_id"]).to_csv(cfg.DATA_PROCESSED / "y_test.csv", header=[cfg.LABEL])
joblib.dump(pre, cfg.MODELS / "preprocessor.joblib")
(cfg.DATA_PROCESSED / "feature_names.json").write_text(json.dumps({"output_features": names, "numeric": num_features, "categorical": cat_features, "log1p": skewed}, indent=2), encoding="utf-8")
print("Saved", A_tr.shape, A_te.shape)